# Solutions — Data Cleaning: Columns, Missing Values, Mixed Dates, Duplicates, Pipelines

Exercise notebook: `09_pandas/06_data_cleaning.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Standardize column names

**Working solution**

In [ ]:
import re

def _snake(name):
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", str(name).strip()))
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

def standardize_columns(df):
    new = [_snake(c) for c in df.columns]
    if len(set(new)) != len(new):
        raise ValueError(f"column names collide after standardizing: {new}")
    return df.set_axis(new, axis=1)

**Explanation**

`set_axis(..., axis=1)` returns a renamed copy in one call. Detecting collisions prevents silently overwriting a column later.

**Complexity:** O(columns).

**Common mistakes**

- `df.columns = ...` mutates the caller's frame.

## Exercise 2 — 🟡 Medium · Fill missing values by type

**Working solution**

In [ ]:
import pandas as pd

def fill_missing(df):
    out, report = df.copy(), {}
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            n = int(s.isna().sum())
            if n:
                out[col] = s.fillna(s.median())
        elif pd.api.types.is_string_dtype(s) or s.dtype == object:
            blanks = s.isna() | s.astype(str).str.strip().eq("")
            n = int(blanks.sum())
            if n:
                out[col] = s.where(~blanks, "UNKNOWN")
        else:
            n = 0
        if n:
            report[col] = n
    return out, report

**Explanation**

Dispatch on dtype with `pd.api.types`. Imputation choices are business decisions — returning a report makes them visible instead of silent.

**Complexity:** O(n × columns).

**Common mistakes**

- Filling with the mean on skewed data (one ₹1,20,000 laptop drags it).
- `fillna('UNKNOWN')` on a numeric column turns it into object dtype.

## Exercise 3 — 🟡 Medium · Normalize-then-dedupe

**Working solution**

In [ ]:
import pandas as pd

def dedupe_customers(df):
    out = df.assign(email_norm=df["email"].astype("string").str.strip().str.lower().replace("", pd.NA))
    has_email = out["email_norm"].notna()
    with_email = (out.loc[has_email].sort_values("signup_date", na_position="last", kind="stable")
                     .drop_duplicates(subset="email_norm", keep="first"))
    return pd.concat([with_email, out.loc[~has_email]]).sort_values("customer_id").reset_index(drop=True)

**Explanation**

Normalize the matching key first — `A@x.io ` and `a@x.io` are the same person. Splitting into 'dedupable' and 'not dedupable' subsets avoids treating all missing emails as one duplicate group.

**Complexity:** O(n log n).

**Common mistakes**

- `drop_duplicates('email')` treats every NaN email as the same value → keeps only one.

## Exercise 4 — 🟠 Hard · Parse mixed date formats vectorized

**Working solution**

In [ ]:
import pandas as pd

def parse_mixed_dates(s):
    text = s.astype("string").str.strip()
    iso = pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")
    return iso.combine_first(dmy)

**Explanation**

Strict formats per pass + `combine_first` (take the first non-null) is deterministic. Letting pandas *guess* (`dayfirst`/inference) silently swaps day and month for dates like `03/04/2025`.

**Complexity:** O(n) per format.

**Common mistakes**

- `pd.to_datetime(s)` with inference on mixed formats → wrong months, no error.

## Challenge — 🔴 Very Hard · A .pipe() cleaning pipeline with a report

**Working solution**

In [ ]:
import re
import pandas as pd

_EMAIL = r"[^@\s]+@[^@\s]+\.[^@\s]+"

def _std_cols(df):
    return df.set_axis([re.sub(r"[^0-9a-z]+", "_", str(c).strip().lower()).strip("_") for c in df.columns], axis=1)

def _strip_and_case(df):
    out = df.apply(lambda s: s.str.strip())
    return out.assign(name=out["name"].str.split().str.join(" ").str.title(), country=out["country"].str.upper(),
                      email=out["email"].str.lower().replace("", pd.NA))

def _types(df, report):
    raw = df["signup_date"]
    parsed = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce").combine_first(pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce"))
    report["unparseable_dates"] = int((parsed.isna() & raw.ne("")).sum())
    return df.assign(signup_date=parsed, is_active=df["is_active"].str.lower().eq("true"))

def _emails(df, report):
    invalid = df["email"].notna() & ~df["email"].fillna("").str.fullmatch(_EMAIL)
    report["invalid_emails"] = int(invalid.sum())
    out = df.assign(email=df["email"].mask(invalid))
    report["missing_emails"] = int(out["email"].isna().sum())
    return out

def _dedupe(df, report):
    has = df["email"].notna()
    kept = df.loc[has].sort_values("signup_date", na_position="last", kind="stable").drop_duplicates("email", keep="first")
    out = pd.concat([kept, df.loc[~has]]).sort_values("customer_id").reset_index(drop=True)
    report["duplicates_removed"] = int(len(df) - len(out))
    return out

def clean_customers(path):
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    report = {"rows_in": int(len(raw))}
    clean = (raw.pipe(_std_cols)
                .pipe(_strip_and_case)
                .pipe(_types, report=report)
                .pipe(_emails, report=report)
                .pipe(_dedupe, report=report))
    report["rows_out"] = int(len(clean))
    report["countries"] = {k: int(v) for k, v in clean["country"].value_counts().sort_index().items()}
    return clean, report

**Explanation**

`.pipe(func, **kwargs)` lets you chain your own functions like built-in methods, so the pipeline reads as a list of named steps. Each step is independently testable, and a shared
report dict collects data-quality metrics as you go. This is the shape of a production pandas job (and of a Glue job with DynamicFrames).

**Complexity:** O(n log n).

**Common mistakes**

- One 200-line function doing everything.
- Mutating the frame in place across steps (hard to debug, breaks under copy-on-write).